# Entrenamiento y Evaluación de Modelos

Entrena y evalúa modelos de clasificación para predecir el comportamiento de precios (Alza/Estable/Caída).

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report 
)
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

## Carga del dataset preprocesado

In [ ]:
df_crudo = pd.read_csv('data/processed/dataset_crudo_sipa.csv')
resultado_preproc = preprocesar_datos(df_crudo)
df = resultado_preproc['dataset_final']
le_producto = resultado_preproc['le_producto']
le_provincia = resultado_preproc['le_provincia']

print(f'Dataset cargado: {len(df)} registros, {df["producto"].nunique()} productos, {df["provincia"].nunique()} provincias')
print(f'Columnas: {list(df.columns)}')
print(f'\nDistribución de comportamiento:')
print(df['comportamiento'].value_counts())
df.head()

## Paso 1 — Limpieza de filas sin historia suficiente (NaN por rezago)

Eliminar filas donde `precio_t2`, `variacion_t2_t1`, `promedio_movil_2q` o `promedio_movil_3q` sean NaN. Estas filas no tienen historia suficiente para generar features de rezago.

In [ ]:
features_rezago = ['precio_t2', 'variacion_t2_t1', 'promedio_movil_2q', 'promedio_movil_3q']

print('Filas con NaN en features de rezago:')
for col in features_rezago:
    n_nan = df[col].isna().sum()
    print(f'  {col}: {n_nan} NaN ({n_nan/len(df)*100:.1f}%)')

filas_antes = len(df)
df_limpio = df.dropna(subset=features_rezago).copy()
filas_despues = len(df_limpio)
filas_eliminadas = filas_antes - filas_despues

print(f'\n--- Resumen limpieza ---')
print(f'Filas antes:        {filas_antes}')
print(f'Filas eliminadas:   {filas_eliminadas} ({filas_eliminadas/filas_antes*100:.1f}%)')
print(f'Filas disponibles:  {filas_despues}')
print(f'\nDistribución después de limpieza:')
print(df_limpio['comportamiento'].value_counts())

## Paso 2 — Variables predictoras y target

In [ ]:
FEATURES = [
    'precio_t1', 'precio_t2', 'variacion_t2_t1',
    'promedio_movil_2q', 'promedio_movil_3q',
    'mes', 'producto_encoded', 'provincia_encoded'
]
TARGET = 'comportamiento'

X = df_limpio[FEATURES].copy()
y = df_limpio[TARGET].copy()

# Codificar target a numérico para XGBoost
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)
print(f'Mapeo target: {dict(zip(le_target.classes_, range(len(le_target.classes_))))}')
print(f'Features shape: {X.shape}')
print(f'Target shape: {y_encoded.shape}')
print(f'\nNaN restantes en X:')
print(X.isna().sum())

## Paso 3 — División temporal (TimeSeriesSplit)

Los datos son series de tiempo: no se puede usar shuffle aleatorio. Se usa `TimeSeriesSplit` con 5 splits.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

print('Divisiones temporales:')
for i, (train_idx, test_idx) in enumerate(tscv.split(X)):
    print(f'  Fold {i+1}: train={len(train_idx)} filas, test={len(test_idx)} filas')
    print(f'           train periodos: {df_limpio.iloc[train_idx]["periodo"].min()} -> {df_limpio.iloc[train_idx]["periodo"].max()}')
    print(f'           test  periodos: {df_limpio.iloc[test_idx]["periodo"].min()} -> {df_limpio.iloc[test_idx]["periodo"].max()}')

## Paso 4 — Entrenamiento de los 6 modelos

Se entrenan los 6 modelos bajo las mismas condiciones (mismas features, mismo split temporal).

In [ ]:


# Se añaden los parámetros de class_weight a los modelos compatibles
modelos_config = {
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1, class_weight='balanced'),
    'XGBoost': XGBClassifier(learning_rate=0.05, max_depth=10, n_estimators=200, tree_method='hist', random_state=42, eval_metric='mlogloss', n_jobs=-1),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, min_samples_split=5, random_state=42, class_weight='balanced'),
    'Logistic Regression': LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, multi_class='multinomial', random_state=42, class_weight='balanced'),
    'KNN': KNeighborsClassifier(n_neighbors=5, weights='distance'),
    'SVM': SVC(C=1.0, kernel='rbf', random_state=42, class_weight='balanced'),
}

resultados = {}

for nombre, modelo_base in modelos_config.items():
    print(f'\n{"="*50}')
    print(f'  {nombre}')
    print(f'{"="*50}')

    fold_metrics = []
    fold_matrices = []

    for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

        modelo = type(modelo_base)(**modelo_base.get_params())

        # Inyección dinámica de pesos para XGBoost (ya que no soporta class_weight directo en multiclase)
        if nombre == 'XGBoost':
            pesos_train = compute_sample_weight('balanced', y_train)
            modelo.fit(X_train, y_train, sample_weight=pesos_train)
        else:
            modelo.fit(X_train, y_train)

        y_pred = modelo.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='macro')
        prec = precision_score(y_test, y_pred, average='macro')
        rec = recall_score(y_test, y_pred, average='macro')
        cm = confusion_matrix(y_test, y_pred)

        fold_metrics.append({'accuracy': acc, 'f1_macro': f1, 'precision': prec, 'recall': rec})
        fold_matrices.append(cm)

        print(f'  Fold {fold+1}: acc={acc:.4f} | f1={f1:.4f} | prec={prec:.4f} | rec={rec:.4f}')

    metrics_df = pd.DataFrame(fold_metrics)
    promedios = metrics_df.mean().to_dict()
    cm_ultimo = fold_matrices[-1]

    resultados[nombre] = {
        'metricas': promedios,
        'metricas_por_fold': metrics_df,
        'matriz_confusion': cm_ultimo,
        'modelo_entrenado': modelo,
        'feature_importances': modelo.feature_importances_ if hasattr(modelo, 'feature_importances_') else None,
    }

    print(f'\n  PROMEDIO: acc={promedios["accuracy"]:.4f} | f1={promedios["f1_macro"]:.4f} | prec={promedios["precision"]:.4f} | rec={promedios["recall"]:.4f}')

## Paso 5 — Evaluación y métricas comparativas

In [ ]:
# Tabla comparativa
tabla = pd.DataFrame({
    nombre: res['metricas'] for nombre, res in resultados.items()
}).T
tabla = tabla[['accuracy', 'f1_macro', 'precision', 'recall']]
tabla.columns = ['Accuracy', 'F1-Score (Macro)', 'Precision (Macro)', 'Recall (Macro)']
tabla = tabla.round(4)

print('=== TABLA COMPARATIVA DE MODELOS ===\n')
print(tabla.to_string())
print(f'\nCriterios de selección: F1-Score >= 0.75 AND Accuracy >= 0.80')

for nombre, res in resultados.items():
    cumple_f1 = res['metricas']['f1_macro'] >= 0.75
    cumple_acc = res['metricas']['accuracy'] >= 0.80
    estado = 'CUMPLE' if (cumple_f1 and cumple_acc) else 'NO CUMPLE'
    print(f'  {nombre}: {estado} (F1={res["metricas"]["f1_macro"]:.4f}, Acc={res["metricas"]["accuracy"]:.4f})')

In [ ]:
# Matrices de confusión
n_modelos = len(resultados)
n_cols = 3
n_rows = (n_modelos + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
axes = axes.flatten() if n_modelos > 1 else [axes]

for idx, (nombre, res) in enumerate(resultados.items()):
    cm = res['matriz_confusion']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=le_target.classes_, yticklabels=le_target.classes_)
    axes[idx].set_title(f'{nombre}')
    axes[idx].set_xlabel('Predicho')
    axes[idx].set_ylabel('Real')

# Ocultar ejes vacíos
for idx in range(n_modelos, len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

## Paso 7 — Análisis: Importancia de variables

In [ ]:
modelos_con_importancia = {k: v for k, v in resultados.items() if v['feature_importances'] is not None}
n_imp = len(modelos_con_importancia)
if n_imp > 0:
    fig, axes = plt.subplots(1, n_imp, figsize=(7 * n_imp, 6))
    axes = axes if n_imp > 1 else [axes]
    for idx, (nombre, res) in enumerate(modelos_con_importancia.items()):
        importancias = pd.Series(res['feature_importances'], index=FEATURES).sort_values(ascending=True)
        importancias.plot(kind='barh', ax=axes[idx], color=sns.color_palette('viridis', len(FEATURES)))
        axes[idx].set_title(f'Importancia de Variables - {nombre}')
        axes[idx].set_xlabel('Importancia')
    plt.tight_layout()
    plt.show()
else:
    print('Ningún modelo tiene feature_importances_ disponible.')

## Selección del modelo final

Criterios: F1-Score (Macro) >= 0.75 AND Accuracy >= 0.80. Si ambos cumplen, se selecciona el que tenga menor cantidad de falsos positivos/negativos.

In [ ]:
candidatos = []
for nombre, res in resultados.items():
    f1 = res['metricas']['f1_macro']
    acc = res['metricas']['accuracy']
    cm = res['matriz_confusion']
    falsos = cm.sum() - np.trace(cm)  # falsos positivos + falsos negativos
    cumple = f1 >= 0.75 and acc >= 0.80
    candidatos.append({
        'Modelo': nombre,
        'F1-Score': f1,
        'Accuracy': acc,
        'Falsos (total)': falsos,
        'Cumple criterios': 'SI' if cumple else 'NO'
    })

df_candidatos = pd.DataFrame(candidatos)
print('=== MODELOS CANDIDATOS ===\n')
print(df_candidatos.to_string(index=False))

# Seleccionar mejor modelo
candidatos_validos = [c for c in candidatos if c['Cumple criterios'] == 'SI']
if candidatos_validos:
    mejor = min(candidatos_validos, key=lambda x: x['Falsos (total)'])
    print(f'\n>> MODELO SELECCIONADO: {mejor["Modelo"]}')
    print(f'   F1-Score: {mejor["F1-Score"]:.4f} | Accuracy: {mejor["Accuracy"]:.4f} | Falsos: {mejor["Falsos (total)"]}')
else:
    # Si ninguno cumple, seleccionar el de mayor F1
    mejor = max(candidatos, key=lambda x: x['F1-Score'])
    print(f'\n>> NINGUNO CUMPLE LOS CRITERIOS. Mejor por F1-Score: {mejor["Modelo"]}')
    print(f'   F1-Score: {mejor["F1-Score"]:.4f} | Accuracy: {mejor["Accuracy"]:.4f}')

## Análisis cualitativo

In [ ]:
# Análisis de la matriz de confusión del mejor modelo
mejor_nombre = mejor['Modelo']
cm_mejor = resultados[mejor_nombre]['matriz_confusion']
clases = le_target.classes_

print(f'=== ANÁLISIS DE MATRIZ DE CONFUSIÓN - {mejor_nombre} ===\n')
print('Matriz de confusión (último fold):\n')
cm_df = pd.DataFrame(cm_mejor, index=[f'Real: {c}' for c in clases], columns=[f'Pred: {c}' for c in clases])
print(cm_df.to_string())

# Diagonal = correctos
correctos = np.trace(cm_mejor)
total = cm_mejor.sum()
print(f'\nCorrectos: {correctos}/{total} ({correctos/total*100:.1f}%)')

# Errores más comunes
print('\nErrores más comunes:')
for i, real in enumerate(clases):
    for j, pred in enumerate(clases):
        if i != j and cm_mejor[i][j] > 0:
            print(f'  Real={real} -> Pred={pred}: {cm_mejor[i][j]} veces')

## Guardado del modelo y artefactos

In [ ]:
os.makedirs('data/models', exist_ok=True)

# Mejor modelo entrenado
modelo_final = resultados[mejor['Modelo']]['modelo_entrenado']
joblib.dump(modelo_final, 'data/models/mejor_modelo.pkl')

# Encoders
joblib.dump(le_target, 'data/models/le_target.pkl')
joblib.dump(le_producto, 'data/models/le_producto.pkl')
joblib.dump(le_provincia, 'data/models/le_provincia.pkl')

# Lista de features (referencia)
joblib.dump(FEATURES, 'data/models/features.pkl')

print('Artefactos guardados en data/models/:')
for f in sorted(os.listdir('data/models/')):
    size = os.path.getsize(f'data/models/{f}')
    print(f'  {f} ({size:,} bytes)')

print(f'\nModelo: {mejor["Modelo"]}')
print(f'F1-Score: {mejor["F1-Score"]:.4f} | Accuracy: {mejor["Accuracy"]:.4f}')